# Loss functions in deep learning
The numbers of the Note: loss versus cost, MSE, MAE, Huber, binary and categorical cross-entropy, sparse categorical
cross-entropy, each by hand with NumPy and then with Keras. The last part fits a line to data with outliers using
three different losses.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import keras
keras.utils.set_random_seed(0)

## Loss of one row, cost of all rows

In [ ]:
# four students: actual package and the network's prediction (LPA)
y = np.array([6.3, 4.1, 3.5, 7.2])
y_hat = np.array([6.1, 4.0, 3.7, 7.0])
err = y - y_hat
print("errors:", err.round(2), "plain sum:", err.sum().round(2))
loss_per_row = err ** 2                      # loss: one value per student
cost = loss_per_row.mean()                   # cost: the average over the batch
print("losses:", loss_per_row.round(4), "cost:", round(cost, 4))
# Keras reports the cost (the mean over the batch)
print("Keras MSE:", float(keras.losses.MeanSquaredError()(y, y_hat)))

## How much one row pushes the weights: the slope of the loss
The update of every weight is proportional to dL/dy_hat. For MSE it is -2(y - y_hat), for MAE it is -sign(y - y_hat).

In [ ]:
for yy, pp in [(8.0, 7.0), (50.0, 8.0)]:     # a normal student and the 50-lakh outlier
    e = yy - pp
    print(f"error {e:5.1f}: MSE loss {e**2:7.1f}, slope {-2*e:6.1f} | MAE loss {abs(e):5.1f}, slope {-np.sign(e):4.1f}")

## Huber loss

In [ ]:
def huber(e, delta=1.0):
    a = np.abs(e)
    # squared (with 1/2) for small errors, straight line for large ones
    return np.where(a <= delta, 0.5 * e ** 2, delta * (a - 0.5 * delta))

for e in [0.5, 1.0, 3.0, 42.0]:
    print(e, float(huber(e)))
print("Keras Huber:", float(keras.losses.Huber(delta=1.0)(np.array([50.0]), np.array([8.0]))))

## Binary cross-entropy

In [ ]:
def bce(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

print(round(bce(1, 0.73), 3), round(bce(0, 0.25), 3))           # natural log
print("with log base 10:", round(-np.log10(0.73), 3), round(-np.log10(0.75), 3))
print("Keras:", [round(float(keras.losses.BinaryCrossentropy()(np.array([t]), np.array([p]))), 3)
                for t, p in [(1.0, 0.73), (0.0, 0.25)]])

## Categorical and sparse categorical cross-entropy

In [ ]:
probs = np.array([[0.2, 0.3, 0.5],          # student 1, true class: yes
                  [0.3, 0.6, 0.1]])         # student 2, true class: no
one_hot = np.array([[1, 0, 0], [0, 1, 0]])
labels = np.array([0, 1])                   # the same classes as integers (Keras counts from 0)
cce = -(one_hot * np.log(probs)).sum(axis=1)
sparse = -np.log(probs[np.arange(2), labels])
print("CCE per row:", cce.round(3), "sparse per row:", sparse.round(3))
print("Keras CCE:", float(keras.losses.CategoricalCrossentropy()(one_hot, probs)),
      "Keras sparse:", float(keras.losses.SparseCategoricalCrossentropy()(labels, probs)),
      "mean by hand:", cce.mean())

## Which output layer goes with which loss

In [ ]:
def model_for(out_units, out_act, loss):
    m = keras.Sequential([keras.Input(shape=(2,)), keras.layers.Dense(3, activation="relu"),
                          keras.layers.Dense(out_units, activation=out_act)])
    m.compile(optimizer="sgd", loss=loss)
    return m

regression = model_for(1, "linear", "mse")          # or "mae", keras.losses.Huber()
binary = model_for(1, "sigmoid", "binary_crossentropy")
multi = model_for(3, "softmax", "categorical_crossentropy")      # one-hot labels
multi_sparse = model_for(3, "softmax", "sparse_categorical_crossentropy")   # integer labels
for m in (regression, binary, multi, multi_sparse):
    print(m.layers[-1].units, m.layers[-1].activation.__name__, m.loss)

## A line through data with outliers: MSE, MAE and Huber
75% of the points follow y = 2x + 1; 25% sit 8 units above. We fit a line with each loss.

In [ ]:
from scipy.optimize import minimize

rng = np.random.default_rng(1)
x = rng.uniform(0, 10, 40)
y_line = 2 * x + 1 + rng.normal(0, 0.5, 40)
is_out = np.arange(40) >= 30                # the last 10 points (25%) are outliers
y_line[is_out] += 8
losses = {"MSE": lambda e: e ** 2, "MAE": lambda e: np.abs(e), "Huber": lambda e: huber(e, 1.0)}
fits = {}
for name, f in losses.items():
    res = minimize(lambda p: f(y_line - (p[0] * x + p[1])).mean(), x0=[0.0, 0.0], method="Nelder-Mead",
                   options=dict(xatol=1e-8, fatol=1e-10, maxiter=5000))
    fits[name] = res.x
    print(name, "slope %.3f intercept %.3f" % tuple(res.x))
np.savez("data/outlier_fit.npz", x=x, y=y_line, is_out=is_out, **{k: v for k, v in fits.items()})